In [47]:
import pandas as pd

df = pd.read_csv(
    "dga-dataset.txt",
    names=["class", "family", "domain"],
    delimiter=","
)

print(df.head())
print(df["class"].value_counts())










   class   family                            domain
0    dga     gozi            mortiscontrastatim.com
1    dga  corebot  cvyh1po636avyrsxebwbkn7.ddns.net
2  legit    alexa                plasticbags.sa.com
3  legit    alexa                      mzltrack.com
4  legit    alexa                      miss-slim.ru
class
dga      337500
legit    337398
Name: count, dtype: int64


In [48]:
df["tld"] = df["domain"].str.split(".").str[-1]








In [49]:
top10_dga_tlds = (
    df[df["class"] == "dga"]["tld"]
    .value_counts()
    .head(10)
)

print("Top 10 TLDs used by DGA domains:")
print(top10_dga_tlds)







Top 10 TLDs used by DGA domains:
tld
com     112931
net      60023
org      28554
ru       21482
eu       16683
info     15689
biz      12306
cc        7099
top       4478
pw        3296
Name: count, dtype: int64


In [50]:
top10_legit_tlds = (
    df[df["class"] == "legit"]["tld"]
    .value_counts()
    .head(10)
)

print("\nTop 10 TLDs used by Legitimate domains:")
print(top10_legit_tlds)







Top 10 TLDs used by Legitimate domains:
tld
com    171217
net     16173
org     15550
ru      13485
de      11065
uk       6580
br       5774
jp       5696
it       4961
ir       4807
Name: count, dtype: int64


In [54]:
# counts you already computed
dga_counts = df[df["class"] == "dga"]["tld"].value_counts()
benign_counts = df[df["class"] == "legit"]["tld"].value_counts()

# align indexes safely
tld_table = pd.concat([dga_counts, benign_counts], axis=1)
tld_table.columns = ["dga", "benign"]

# fill missing with 0
tld_table = tld_table.fillna(0)

# association score
tld_table["dga_ratio"] = tld_table["dga"] / (tld_table["dga"] + tld_table["benign"])














In [55]:
tld_table.sort_values("dga_ratio", ascending=False).head(10)





,dga,benign,dga_ratio
tld,,,
tel,194.0,0.0,1.000000
kn,120.0,0.0,1.000000
bs,121.0,0.0,1.000000
rent,6.0,0.0,1.000000
nf,435.0,0.0,1.000000
dm,108.0,0.0,1.000000
bit,319.0,0.0,1.000000
ki,594.0,1.0,0.998319
hn,466.0,3.0,0.993603


In [58]:
tld_table["total"] = tld_table["dga"] + tld_table["benign"]

# keep only reliable TLDs
tld_table_f = tld_table[tld_table["total"] >= 1000]









In [59]:
tld_table_f["dga_ratio"] = (
    tld_table_f["dga"] / tld_table_f["total"]
)






C:\Users\SİNAN\AppData\Local\Temp\ipykernel_15984\2918283802.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tld_table_f["dga_ratio"] = (


In [60]:
tld_table_f.sort_values("dga_ratio", ascending=False).head(10)





,dga,benign,dga_ratio,total
tld,,,,
top,4478.0,355.0,0.926547,4833.0
pw,3296.0,268.0,0.924804,3564.0
cc,7099.0,609.0,0.920991,7708.0
biz,12306.0,1069.0,0.920075,13375.0
su,2840.0,257.0,0.917016,3097.0
eu,16683.0,1741.0,0.905504,18424.0
info,15689.0,3323.0,0.825216,19012.0
net,60023.0,16173.0,0.787745,76196.0
me,2550.0,1188.0,0.682183,3738.0


In [61]:
tld_table_f.sort_values("dga_ratio").head(10)





,dga,benign,dga_ratio,total
tld,,,,
au,0.0,2390.0,0.000000,2390.0
io,0.0,1333.0,0.000000,1333.0
se,0.0,1107.0,0.000000,1107.0
br,117.0,5774.0,0.019861,5891.0
fr,135.0,4161.0,0.031425,4296.0
es,104.0,2931.0,0.034267,3035.0
it,190.0,4961.0,0.036886,5151.0
nl,116.0,1954.0,0.056039,2070.0
ca,116.0,1666.0,0.065095,1782.0


In [62]:
alpha = 1  # Laplace smoothing
tld_table_f["dga_ratio"] = (
    (tld_table_f["dga"] + alpha) /
    (tld_table_f["total"] + 2 * alpha)
)




C:\Users\SİNAN\AppData\Local\Temp\ipykernel_15984\599155808.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tld_table_f["dga_ratio"] = (


NameError: name 'df_benign' is not defined